<a href="https://colab.research.google.com/github/gowtham-garimella/AI-Smart-Surveillance-System-YOLOv8-Real-Time-Dashboard-/blob/main/AI_Smart_Surveillance_System_(YOLOv8_%2B_Real_Time_Dashboard).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ================= INSTALL =================
!pip install ultralytics opencv-python pandas yt-dlp --quiet

# ================= IMPORTS =================
import cv2
import pandas as pd
from ultralytics import YOLO
from datetime import datetime
import urllib.request
import os

# ================= SETTINGS =================
CONF_THRESHOLD = float(input("🎯 Enter confidence threshold (e.g., 0.5): ") or 0.5)
ALERT_CLASSES = input("🚨 Enter alert classes (comma-separated, e.g., person,car): ") or "person"
FRAME_SKIP = int(input("⚡ Enter frame skip (e.g., 2 for faster processing): ") or 1)

ALERT_CLASSES = [c.strip().lower() for c in ALERT_CLASSES.split(",")]

# ================= LOAD MODEL =================
model = YOLO("yolov8n.pt")

# ================= DATABASE =================
db_logs = []

def store_detection(detections):
    db_logs.extend(detections)

# ================= ALERT SYSTEM =================
def send_alert(detection):
    if detection["label"].lower() in ALERT_CLASSES:
        print(f"🚨 ALERT: {detection['label']} | Camera: {detection['camera_id']} | Time: {detection['timestamp']}")

# ================= BACKEND SIM =================
def backend_receive(detections):
    store_detection(detections)
    for d in detections:
        send_alert(d)

# ================= VIDEO INPUT =================
video_input = input("🔗 Enter video link (MP4 or YouTube): ") or "https://assets.mixkit.co/videos/preview/mixkit-pedestrians-crossing-the-street-at-a-crosswalk-4484-large.mp4"

video_path = "input_video.mp4"

# Detect YouTube vs direct link
if "youtube.com" in video_input or "youtu.be" in video_input:
    print("⬇️ Downloading from YouTube...")
    !yt-dlp -o input_video.mp4 "{video_input}"
else:
    print("⬇️ Downloading direct video...")
    urllib.request.urlretrieve(video_input, video_path)

print("✅ Download complete!")

# ================= VIDEO SETUP =================
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 640)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 480)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, 20.0, (width, height))

# ================= PROCESS STREAM =================
camera_id = "cam_1"
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Skip frames for speed
    if frame_count % FRAME_SKIP != 0:
        continue

    results = model(frame)[0]
    detections = []

    for box in results.boxes:
        cls = int(box.cls[0])
        label = model.names[cls]
        conf = float(box.conf[0])

        if conf >= CONF_THRESHOLD:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Draw box
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, f"{label} {conf:.2f}", (x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

            detection = {
                "label": label,
                "confidence": conf,
                "timestamp": str(datetime.now()),
                "camera_id": camera_id
            }

            detections.append(detection)

    if detections:
        backend_receive(detections)

    out.write(frame)

cap.release()
out.release()

# ================= SAVE LOGS =================
df = pd.DataFrame(db_logs)
df.to_csv("detection_logs.csv", index=False)

print("\n✅ Processing Complete!")
print("📁 Files: output.mp4, detection_logs.csv")

🎯 Enter confidence threshold (e.g., 0.5): 0.6
🚨 Enter alert classes (comma-separated, e.g., person,car): car
⚡ Enter frame skip (e.g., 2 for faster processing): 3
🔗 Enter video link (MP4 or YouTube): https://www.youtube.com/watch?v=MNn9qKG2UFI
⬇️ Downloading from YouTube...
[youtube] Extracting URL: https://www.youtube.com/watch?v=MNn9qKG2UFI
[youtube] MNn9qKG2UFI: Downloading webpage
[youtube] MNn9qKG2UFI: Downloading android vr player API JSON
[info] MNn9qKG2UFI: Downloading 1 format(s): 313+251
[download] Destination: input_video.mp4.f313.webm
[download] 100% of  481.87MiB in 00:00:26 at 17.86MiB/s
[download] Destination: input_video.mp4.f251.webm
[download] 100% of    4.32MiB in 00:00:00 at 20.63MiB/s
[Merger] Merging formats into "input_video.mp4.webm"
Deleting original file input_video.mp4.f313.webm (pass -k to keep)
Deleting original file input_video.mp4.f251.webm (pass -k to keep)
✅ Download complete!

✅ Processing Complete!
📁 Files: output.mp4, detection_logs.csv


In [9]:
# ================= INSTALL =================
!pip install ultralytics opencv-python pandas yt-dlp --quiet

# ================= IMPORTS =================
import cv2
import pandas as pd
from ultralytics import YOLO
from datetime import datetime
import urllib.request
from IPython.display import display, HTML
import base64

# ================= SETTINGS =================
CONF_THRESHOLD = 0.5
FRAME_SKIP = 2
MAX_FRAMES = 200   # 🔥 ensures fast completion
ALERT_CLASSES = ["person", "car"]

# ================= LOAD MODEL =================
model = YOLO("yolov8n.pt")

# ================= VIDEO INPUT =================
video_input = input("🔗 Enter video link (MP4 or YouTube): ") or "https://videos.pexels.com/video-files/6883271/6883271-sd_640_360_25fps.mp4"
video_path = "input_video.mp4"

if "youtube.com" in video_input or "youtu.be" in video_input:
    print("⬇️ Downloading YouTube video...")
    !yt-dlp -f mp4 -o input_video.mp4 "{video_input}"
else:
    print("⬇️ Downloading MP4 video...")
    urllib.request.urlretrieve(video_input, video_path)

print("✅ Download complete!")

# ================= VIDEO SETUP =================
cap = cv2.VideoCapture(video_path)

width, height = 480, 320  # optimized for display
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, 20.0, (width, height))

# ================= STORAGE =================
logs = []
alerts = []
frame_count = 0

# ================= PROCESS =================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_count >= MAX_FRAMES:
        break

    frame_count += 1

    if frame_count % FRAME_SKIP != 0:
        continue

    frame = cv2.resize(frame, (width, height))
    results = model(frame)[0]

    for box in results.boxes:
        label = model.names[int(box.cls[0])]
        conf = float(box.conf[0])

        if conf >= CONF_THRESHOLD:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Draw box
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, f"{label}", (x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

            timestamp = datetime.now().strftime("%H:%M:%S")

            logs.append({
                "time": timestamp,
                "label": label,
                "confidence": conf
            })

            if label.lower() in ALERT_CLASSES:
                alerts.append(f"{label} detected at {timestamp}")

    out.write(frame)

cap.release()
out.release()

print("✅ Processing Finished!")

# ================= VIDEO DISPLAY =================
video_bytes = open("output.mp4", "rb").read()
video_base64 = base64.b64encode(video_bytes).decode()

# ================= STATS =================
df = pd.DataFrame(logs)
counts = df['label'].value_counts().to_dict() if not df.empty else {}

# ================= DASHBOARD UI =================
html = f"""
<style>
body {{ background-color: #0b0f1a; color: white; font-family: Arial; }}
.box {{ background: #1c2333; padding: 10px; margin: 10px 0; border-radius: 10px; }}
h2 {{ color: #00ffcc; }}
</style>

<h2>🛡️ AI Smart Surveillance Dashboard</h2>

<div class="box">
<h3>🎥 Camera Feed</h3>
<video width="500" controls>
  <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
</video>
</div>

<div class="box">
<h3>🚨 Alerts</h3>
{"<br>".join(alerts[-10:]) if alerts else "No alerts"}
</div>

<div class="box">
<h3>📊 Detection Stats</h3>
{"<br>".join([f"{k}: {v}" for k,v in counts.items()]) if counts else "No data"}
</div>

<div class="box">
<h3>🧾 Recent Logs</h3>
{"<br>".join([f"{l['time']} - {l['label']} ({l['confidence']:.2f})" for l in logs[-15:]]) if logs else "No logs"}
</div>
"""

display(HTML(html))

🔗 Enter video link (MP4 or YouTube): https://www.youtube.com/watch?v=MNn9qKG2UFI
⬇️ Downloading YouTube video...
         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[youtube] Extracting URL: https://www.youtube.com/watch?v=MNn9qKG2UFI
[youtube] MNn9qKG2UFI: Downloading webpage
[youtube] MNn9qKG2UFI: Downloading android vr player API JSON
[info] MNn9qKG2UFI: Downloading 1 format(s): 18
[download] input_video.mp4 has already been downloaded
[download] 100% of   20.91MiB
✅ Download complete!

0: 448x640 1 person, 4 cars, 1 bus, 2 trains, 114.7ms
Speed: 2.6ms preprocess, 114.7ms inference, 0.8ms postprocess per image at shape (1, 3, 448, 640)

0: 448x640 1 person, 4 cars, 1 bus, 1 train, 110.9ms
Speed: 2.1ms pr

In [10]:
# ================= INSTALL =================
!pip install ultralytics opencv-python pandas yt-dlp imageio[ffmpeg] --quiet

# ================= IMPORTS =================
import cv2
import pandas as pd
from ultralytics import YOLO
from datetime import datetime
import urllib.request
from IPython.display import display, HTML
import base64

# ================= SETTINGS =================
CONF_THRESHOLD = 0.5
FRAME_SKIP = 2
MAX_FRAMES = 150   # keeps it fast and ensures UI shows
ALERT_CLASSES = ["person", "car"]

# ================= LOAD MODEL =================
model = YOLO("yolov8n.pt")

# ================= VIDEO INPUT =================
video_input = input("🔗 Enter video link (MP4 or YouTube): ")
video_path = "input_video.mp4"

if "youtube.com" in video_input or "youtu.be" in video_input:
    print("⬇️ Downloading from YouTube...")
    !yt-dlp -f mp4 -o input_video.mp4 "{video_input}"
else:
    print("⬇️ Downloading MP4...")
    urllib.request.urlretrieve(video_input, video_path)

print("✅ Download complete!")

# ================= VIDEO SETUP =================
cap = cv2.VideoCapture(video_path)

width, height = 320, 240   # smaller = reliable playback
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_raw.mp4', fourcc, 15.0, (width, height))

# ================= STORAGE =================
logs = []
alerts = []
frame_count = 0

# ================= PROCESS =================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_count >= MAX_FRAMES:
        break

    frame_count += 1

    if frame_count % FRAME_SKIP != 0:
        continue

    frame = cv2.resize(frame, (width, height))
    results = model(frame)[0]

    for box in results.boxes:
        label = model.names[int(box.cls[0])]
        conf = float(box.conf[0])

        if conf >= CONF_THRESHOLD:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, label, (x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

            t = datetime.now().strftime("%H:%M:%S")

            logs.append({
                "time": t,
                "label": label,
                "confidence": conf
            })

            if label.lower() in ALERT_CLASSES:
                alerts.append(f"🚨 {label} detected at {t}")

    out.write(frame)

cap.release()
out.release()

print("🎬 Compressing video...")

# ================= COMPRESS VIDEO =================
!ffmpeg -i output_raw.mp4 -vcodec libx264 -crf 28 output.mp4 -y -loglevel quiet

print("✅ Processing Complete!")

# ================= VIDEO DISPLAY =================
video_bytes = open("output.mp4", "rb").read()
video_base64 = base64.b64encode(video_bytes).decode()

# ================= STATS =================
df = pd.DataFrame(logs)
counts = df['label'].value_counts().to_dict() if not df.empty else {}

# ================= DASHBOARD UI =================
html = f"""
<style>
body {{ background-color: #0b0f1a; color: white; font-family: Arial; }}
.container {{ display: flex; }}
.video {{ flex: 2; }}
.sidebar {{ flex: 1; padding-left: 15px; }}
.box {{ background: #1c2333; padding: 10px; margin-bottom: 10px; border-radius: 10px; }}
h2 {{ color: #00ffcc; }}
</style>

<h2>🛡️ AI Smart Surveillance Dashboard</h2>

<div class="container">
    <div class="video">
        <div class="box">
            <h3>🎥 Camera Feed</h3>
            <video width="100%" controls>
                <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
            </video>
        </div>
    </div>

    <div class="sidebar">
        <div class="box">
            <h3>🚨 Alerts</h3>
            {"<br>".join(alerts[-8:]) if alerts else "No alerts"}
        </div>

        <div class="box">
            <h3>📊 Stats</h3>
            {"<br>".join([f"{k}: {v}" for k,v in counts.items()]) if counts else "No data"}
        </div>
    </div>
</div>

<div class="box">
<h3>🧾 Logs</h3>
{"<br>".join([f"{l['time']} - {l['label']} ({l['confidence']:.2f})" for l in logs[-10:]]) if logs else "No logs"}
</div>
"""

display(HTML(html))

# ================= DOWNLOAD =================
from google.colab import files
files.download("output.mp4")

🔗 Enter video link (MP4 or YouTube): https://www.youtube.com/watch?v=MNn9qKG2UFI
⬇️ Downloading from YouTube...
         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning
[youtube] Extracting URL: https://www.youtube.com/watch?v=MNn9qKG2UFI
[youtube] MNn9qKG2UFI: Downloading webpage
[youtube] MNn9qKG2UFI: Downloading android vr player API JSON
[info] MNn9qKG2UFI: Downloading 1 format(s): 18
[download] input_video.mp4 has already been downloaded
[download] 100% of   20.91MiB
✅ Download complete!

0: 480x640 6 cars, 1 bus, 1 train, 152.0ms
Speed: 4.0ms preprocess, 152.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 cars, 1 bus, 162.2ms
Speed: 3.9ms preprocess, 162.2ms inference, 1.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>